In [ ]:
# --- Imports (PyTorch only; TensorFlow removed) ---------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import itertools
import re
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from gensim.models import Word2Vec

import mlflow
mlflow.set_experiment("keras1")

labelaa = 'Single_Label'
df = pd.read_pickle('springer_dataframe_5_categories.p')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[labelaa], random_state=40)
test_df, val_df   = train_test_split(test_df, test_size=0.5, stratify=test_df[labelaa], random_state=40)

def compute_prototypes(embedding_net, source_df, labels, vector_col, device):
    prototypes = {}
    with torch.no_grad():
        for label in labels:
            class_df = source_df[source_df[labelaa] == label]
            class_vectors = torch.tensor(
                np.stack(class_df[vector_col].values), dtype=torch.float32
            ).to(device)
            class_embeddings = embedding_net(class_vectors)
            prototypes[label] = torch.mean(class_embeddings, dim=0)
    return prototypes


def compute_accuracy(embedding_net, prototypes, eval_df, vector_col, device):
    eval_vectors_np = np.stack(eval_df[vector_col].values)
    eval_vectors = torch.tensor(eval_vectors_np, dtype=torch.float32).to(device)
    eval_labels = eval_df[labelaa].values
    with torch.no_grad():
        eval_embeddings = embedding_net(eval_vectors)
    correct = 0
    for i in range(len(eval_labels)):
        embed = eval_embeddings[i]
        true_label = eval_labels[i]
        best_label = None
        smallest_distance = float('inf')
        for label, prototype in prototypes.items():
            sim = F.cosine_similarity(embed.unsqueeze(0), prototype.unsqueeze(0))
            dist = 1.0 - sim.item()
            if dist < smallest_distance:
                smallest_distance = dist
                best_label = label
        if best_label == true_label:
            correct += 1
    return (correct / len(eval_labels)) * 100


In [ ]:
# --- Text preprocessing + tokenizer ---------------------------------------
# clean_text is unchanged. The Keras Tokenizer is re-implemented in pure
# Python with identical semantics: index 0 = padding, index 1 = <OOV>, real
# words 2.. ranked by frequency and capped at MAX_NUM_WORDS; post-pad/truncate.
def tokenize(text):
    return re.findall(r'\b[a-zA-Z]+\b', str(text).lower())

def clean_text(text):
    return ' '.join(tokenize(text))

train_df['vector'] = train_df['toc'].apply(clean_text)
val_df['vector']   = val_df['toc'].apply(clean_text)
test_df['vector']  = test_df['toc'].apply(clean_text)

lengths = train_df['vector'].apply(lambda x: len(x.split()))
MAX_SEQUENCE_LENGTH = int(lengths.quantile(0.95))
MAX_NUM_WORDS = 20000
print(f"MAX_SEQUENCE_LENGTH = {MAX_SEQUENCE_LENGTH} (95-ti percentil dužine TOC-ova)")

counts = Counter()
for t in train_df['vector']:
    counts.update(t.split())
word_index = {w: i + 2 for i, (w, _) in enumerate(counts.most_common(MAX_NUM_WORDS - 2))}
vocab_size = min(MAX_NUM_WORDS, len(word_index) + 2)
print(f"Veličina vokabulara: {len(counts)} jedinstvenih reči")

def texts_to_padded(texts, maxlen):
    out = np.zeros((len(texts), maxlen), dtype=np.int64)
    for i, t in enumerate(texts):
        seq = [word_index.get(w, 1) for w in t.split()][:maxlen]   # OOV -> 1, post-truncate
        out[i, :len(seq)] = seq                                    # post-pad with 0
    return out

X_train_pad = texts_to_padded(train_df['vector'], MAX_SEQUENCE_LENGTH)
X_val_pad   = texts_to_padded(val_df['vector'],   MAX_SEQUENCE_LENGTH)
X_test_pad  = texts_to_padded(test_df['vector'],  MAX_SEQUENCE_LENGTH)
print(f"Oblik X_train_pad: {X_train_pad.shape}")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df[labelaa])
y_val   = label_encoder.transform(val_df[labelaa])
y_test  = label_encoder.transform(test_df[labelaa])
num_classes = len(label_encoder.classes_)
print(f"Klase: {list(label_encoder.classes_)}")


In [ ]:
# --- BiLSTM document-embedding classifier (PyTorch port of the Keras model) -
# Keras: Embedding(128) -> Bidirectional(LSTM(300)) [doc_vector, 600-dim]
#        -> Dropout(.5) -> Dense(64, relu) -> Dropout(.3) -> Dense(C, softmax)
EMBEDDING_DIM = 128
LSTM_UNITS = 300
DOC_VECTOR_DIM = 2 * LSTM_UNITS            # bidirectional concat -> 600

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, lstm_units, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, lstm_units, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(2 * lstm_units, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def encode(self, x):                   # -> doc_vector: final fwd+bwd hidden states (600-dim)
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        return torch.cat([h_n[-2], h_n[-1]], dim=1)

    def forward(self, x):                  # logits (CrossEntropyLoss applies softmax internally)
        z = self.dropout1(self.encode(x))
        z = torch.relu(self.fc1(z))
        z = self.dropout2(z)
        return self.fc2(z)

clf = BiLSTMClassifier(vocab_size, EMBEDDING_DIM, LSTM_UNITS, num_classes).to(device)
optimizer_clf = torch.optim.Adam(clf.parameters())
criterion_clf = nn.CrossEntropyLoss()

CLS_EPOCHS = 1            # matches the original Keras model.fit(epochs=1)
CLS_BATCH = 32
train_dl = DataLoader(TensorDataset(torch.tensor(X_train_pad), torch.tensor(y_train)),
                      batch_size=CLS_BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(torch.tensor(X_val_pad), torch.tensor(y_val)),
                      batch_size=CLS_BATCH, shuffle=False)

for epoch in range(CLS_EPOCHS):
    clf.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_clf.zero_grad()
        loss = criterion_clf(clf(xb), yb)
        loss.backward()
        optimizer_clf.step()
        running += loss.item() * xb.size(0)
    clf.eval(); correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (clf(xb).argmax(1) == yb).sum().item(); total += yb.size(0)
    print(f"Epoch {epoch+1}/{CLS_EPOCHS} -> train_loss={running/len(train_dl.dataset):.4f} "
          f"val_acc={100*correct/total:.2f}%")

# Extract the doc_vector embeddings (replaces Keras embedding_extractor.predict)
clf.eval()
def extract_doc_vectors(X):
    chunks = []
    with torch.no_grad():
        for i in range(0, len(X), 64):
            xb = torch.tensor(X[i:i+64]).to(device)
            chunks.append(clf.encode(xb).cpu().numpy())
    return np.concatenate(chunks)

train_vectors = extract_doc_vectors(X_train_pad)
val_vectors   = extract_doc_vectors(X_val_pad)
test_vectors  = extract_doc_vectors(X_test_pad)
print(f"\nOblik izvučenih vektora: {train_vectors.shape}")   # (N, 600)

train_df['vector'] = list(train_vectors)
val_df['vector']   = list(val_vectors)
test_df['vector']  = list(test_vectors)

del clf
gc.collect()
torch.cuda.empty_cache()

labels = train_df[labelaa].unique()


In [ ]:
# --- Siamese metric-learning stage (unchanged from the original torch code) -
def build_pairs(source_df, labels):
    pairs_list = []
    for label1, label2 in itertools.combinations_with_replacement(labels, 2):
        df_class1 = source_df[source_df[labelaa] == label1]
        df_class2 = source_df[source_df[labelaa] == label2]
        N = min(len(df_class1), len(df_class2))
        if N == 0:
            continue
        left_samples = df_class1.sample(n=N, replace=True).reset_index(drop=True)
        right_samples = df_class2.sample(n=N, replace=True).reset_index(drop=True)
        is_same = 1 if label1 == label2 else 0
        for i in range(N):
            pairs_list.append({
                'input_left': left_samples.iloc[i]['vector'],
                'input_right': right_samples.iloc[i]['vector'],
                'label_left': label1,
                'label_right': label2,
                'is_same': is_same
            })
    return pairs_list

train_pairs_df = pd.DataFrame(build_pairs(train_df, labels)).sample(frac=1, random_state=42).reset_index(drop=True)
val_pairs_df   = pd.DataFrame(build_pairs(val_df, labels)).sample(frac=1, random_state=42).reset_index(drop=True)


class EmbeddingNetwork(nn.Module):
    # FIX: input_dim is the BiLSTM doc-vector size (600), not EMBEDDING_DIM (128).
    def __init__(self, input_dim=DOC_VECTOR_DIM, embedding_dim=128):
        super(EmbeddingNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, embedding_dim)
        )
    def forward(self, x):
        return self.fc(x)


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_net):
        super(SiameseNetwork, self).__init__()
        self.embedding_net = embedding_net
    def forward(self, x_left, x_right):
        return self.embedding_net(x_left), self.embedding_net(x_right)


class SiameseDataset(Dataset):
    def __init__(self, pairs_df):
        self.x_left = np.stack(pairs_df['input_left'].values)
        self.x_right = np.stack(pairs_df['input_right'].values)
        self.labels = pairs_df['is_same'].values
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.x_left[idx], dtype=torch.float32),
            torch.tensor(self.x_right[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(SiameseDataset(train_pairs_df), batch_size=16, shuffle=True)
val_loader   = DataLoader(SiameseDataset(val_pairs_df), batch_size=16, shuffle=False)


class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=1):
        super(CosineContrastiveLoss, self).__init__()
        self.margin = margin
    def forward(self, output_left, output_right, target):
        cosine_sim = F.cosine_similarity(output_left, output_right)
        cosine_dist = 1 - cosine_sim
        loss_same = target * torch.pow(cosine_dist, 2)
        loss_diff = (1 - target) * torch.pow(torch.clamp(self.margin - cosine_dist, min=0.0), 2)
        return torch.mean(loss_same + loss_diff)

emb_net = EmbeddingNetwork(input_dim=DOC_VECTOR_DIM, embedding_dim=128)
model = SiameseNetwork(emb_net).to(device)

NUM_EPOCHS = 20
LR = 0.005
criterion = CosineContrastiveLoss(margin=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

with mlflow.start_run():
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", NUM_EPOCHS)
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for batch_left, batch_right, batch_labels in train_loader:
            batch_left = batch_left.to(device)
            batch_right = batch_right.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            out_left, out_right = model(batch_left, batch_right)
            loss = criterion(out_left, out_right, batch_labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_left.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        model.eval()
        val_running_loss = 0.0
        with torch.no_grad():
            for val_left, val_right, val_labels in val_loader:
                val_left = val_left.to(device)
                val_right = val_right.to(device)
                val_labels = val_labels.to(device)
                val_out_left, val_out_right = model(val_left, val_right)
                val_loss = criterion(val_out_left, val_out_right, val_labels)
                val_running_loss += val_loss.item() * val_left.size(0)
        epoch_val_loss = val_running_loss / len(val_loader.dataset)
        prototypes = compute_prototypes(emb_net, train_df, labels, 'vector', device)
        val_accuracy = compute_accuracy(emb_net, prototypes, val_df, 'vector', device)
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("val_loss", epoch_val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)
        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] -> Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")
    model.eval()
    final_prototypes = compute_prototypes(emb_net, train_df, labels, 'vector', device)    # FIX: added vector_col='vector'
    test_accuracy = compute_accuracy(emb_net, final_prototypes, test_df, 'vector', device) # FIX: added vector_col='vector'
    mlflow.log_metric("final_accuracy", test_accuracy)
    print("-" * 40)
    print("Evaluation Results:")
    print(f"Total Test Samples: {len(test_df)}")
    print(f"Final Model Accuracy: {test_accuracy:.2f}%")
